# Session 06 Topic 01: From a sample to a claim about a population

Use this notebook while working through Topic 01. It introduces the dataset used for
most of this session — six years of daily boardings on three Melbourne train lines.

The setup and file-loading code is supplied. You will write the analysis code and
answer the activity questions. Work from top to bottom, because later cells use
variables created earlier.

By the end of this notebook you will have calculated a point estimate and seen why,
on its own, it is not enough.

## 1. The dataset for this session

Run the setup cell. It imports the libraries, records where the data file is, and
loads the daily boardings into `train`.

Each row is **one line on one day**. The original archive recorded every individual
train calling at every station — 90.1 million rows — which is far more detail than a
question about daily patronage needs.

In [ ]:
from pathlib import Path

import pandas as pd

DATA_FOLDER = Path("../data")
TRAIN_FILE = DATA_FOLDER / "train_daily_boardings.csv"

train = pd.read_csv(TRAIN_FILE, parse_dates=["business_date"])
train.head()

The six columns are:

| Column | Meaning |
|---|---|
| `business_date` | the calendar day |
| `line_name` | Sandringham, Williamstown, or Alamein |
| `day_of_week` | Monday to Sunday |
| `day_type` | Normal Weekday, School Holiday, Saturday, Sunday, or Public Holiday |
| `financial_year` | for example `2023-24`, running July to June |
| `total_boardings` | everyone who boarded that line that day |

Now answer the three questions from the Activity in Topic 01. Each needs one short
line of pandas.

In [ ]:
print("Days for each line:")
print(train["line_name"].value_counts())

In [ ]:
print("Day types, most common first:")
print(train["day_type"].value_counts())

In [ ]:
print("Earliest date:", train["business_date"].min().date())
print("Latest date:  ", train["business_date"].max().date())
print("Total rows:   ", len(train))

### Activity — Meet the data


How many days does each line have? Which `day_type` values appear most often? What is
the earliest and latest date?

**Answer:** 
The three lines have similar but not identical counts: Alamein 2,166 days,
Williamstown 2,124, and Sandringham 2,123 — 6,413 rows in total. Six financial years
is 2,192 days, so every line is missing a few. Those are days with no recorded
services at all, usually planned shutdowns for track works.

That is worth noticing rather than passing over. A missing day is not a day with zero
boardings; it is a day the line did not run. Leaving them out is right, because
averaging in zeros would drag every summary down and describe something that never
happened.

`Normal Weekday` is much the most common day type at 3,532 rows, followed by
`School Holiday` (918), `Sunday` (857), `Saturday` (855) and `Public Holiday` (251).
This matters for everything that follows: weekends and holidays behave very
differently from ordinary weekdays, so mixing them together would produce averages
describing no real kind of day at all.

## 2. Population and sample

A **population** is every member of the group you want to describe. A **sample** is the
part of it you actually measured.

Which is which depends entirely on the question you asked, not on the file. There is
no code for this section — read Topic 01, then answer below.

### Activity — Name the population


For each question, decide whether the file is a population or a sample, and describe
the population in words.

1. How many people boarded the Alamein line on 14 March 2024?
2. Are Williamstown Saturdays busier than Sundays?
3. Should the Sandringham line run extra Monday services next winter?

**Answer:** 
**1. Population — and no inference is needed at all.** The question is about one
specific day that is recorded in the file. You look it up and you have the exact
answer. A confidence interval here would add uncertainty where none exists.

**2. Sample.** The question is about what Williamstown Saturdays are *like*, not about
the particular 300-odd Saturdays that happen to be recorded. The population is all the
Saturdays that line could run under similar conditions.

**3. Sample, and the population includes days that have not happened yet.** The
question is about next winter. The observed Mondays are useful only as evidence about
the process that generates Mondays — which assumes that process is stable enough for
past Mondays to say something about future ones.

## 3. Parameters and statistics

| | Population | Sample |
|---|---|---|
| What it is | every member | the part you measured |
| Its mean is called | a **parameter** | a **statistic** |
| Written as | $\mu$ (mu) | $\bar{x}$ (x-bar) |
| Do you know it? | almost never | yes — you calculated it |

A parameter is fixed and unknown. A statistic is known and varies from sample to
sample. The whole session is about using the second to make a claim about the first.

## 4. The point estimate

Filter the data to the group we will use throughout this session: the **Sandringham**
line, **normal weekdays**, in financial year **2023-24**.

Then calculate its mean. That single value is the **point estimate** of the population
mean.

In [ ]:
sandringham = train[
    (train["line_name"] == "Sandringham")
    & (train["day_type"] == "Normal Weekday")
    & (train["financial_year"] == "2023-24")
]

print("Days:", len(sandringham))
print("Mean daily boardings:", round(sandringham["total_boardings"].mean()))

You should have **202 days** and a mean of **41,460** boardings.

Keep the `sandringham` filter in mind — it is used in every notebook in this session.

## 5. Why the point estimate is not enough

The sample mean is the best single guess available. But a single number carries no
information about how much it might have moved if different days had been measured.

Compare the full 202 days with a small handful of them.

In [ ]:
boardings = sandringham["total_boardings"]

small_sample = boardings.sample(n=5, random_state=1106)

print(f"All 202 days:      mean = {boardings.mean():,.0f}")
print(f"Just 5 of them:    mean = {small_sample.mean():,.0f}")

The two means are in the same region, but they are not equally trustworthy — and
nothing in either number says so. Run the next cell to see how much the 5-day answer
moves when you take a different 5 days.

In [ ]:
for seed in range(8):
    draw = boardings.sample(n=5, random_state=seed)
    print(f"seed {seed}: mean of 5 days = {draw.mean():,.0f}")

Those eight answers run from about **38,600 to 45,200** — a spread of nearly 6,700
boardings, all of them honest estimates of the same quantity. The true value is 41,460.

Nothing distinguishes a good draw from a bad one at the time you take it. That is the
problem the rest of the session solves.

### Activity — The same number, different confidence


Each campus reports a mean mark of 58. One is based on 5,214 results, the other
on 66. Write one sentence explaining why a manager should treat these differently,
without using any statistical terms.

**Answer:** 
A sample answer:

> The first figure is based on so many students that it would barely shift if a few
> more were added, while the second rests on so few that a handful of unusual results
> could have moved it several marks either way.

The point is that the *reliability* of an average depends on how much evidence sits
behind it, and the average alone does not show that. This is exactly what the cells
above demonstrated: the mean of 5 days ranged from 38,600 to 45,200 depending on which
5 you happened to take, while the mean of all 202 days is pinned at 41,460.

The missing piece has a name — the **margin of error** — and Topics 2 and 3 build it.

## What you have done

- Loaded the daily boardings data and established that one row is one line on one day
- Practised deciding whether a dataset is a population or a sample, and seen that the
  answer depends on the question
- Calculated a point estimate: 41,460 mean weekday boardings on the Sandringham line
- Watched a 5-day estimate swing wildly while the 202-day estimate stayed put

**Next:** Topic 02's notebook measures that swing, and turns it into a number.